In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import re #for filtering and finding all breed types

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIXED #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from crud_animals import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIXED update with your username and password and CRUD Python module name

username = "accuser"
password = "UseThisOne"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
#df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('Project Two App')

#FIXED Add in Grazioso Salvare’s logo
image_filename = 'GraziosoSalvareLogo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIXED Place the HTML image tag in the line below into the app.layout code according to your design
#FIXED Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([  
    #https://www.w3schools.com/tags/tag_a.asp
    html.A([
        html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                            height=125, width =126))],
                            href = 'https://www.snhu.edu',
                            target="_blank" #opens new tab for website (https://stackoverflow.com/questions/31434143/how-to-make-a-html-link-to-open-in-new-tab-in-python)
    ),
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H2('CS-340 Dashboard - K.Rouse'))),
    html.Hr(),
    html.Div(),
#FIXED Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.       
# https://dash.plotly.com/dash-core-components/radioitems
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label':'Water Rescue','value':'Water'},
            {'label':'Mountain or Wilderness Rescue','value':'Mountain'},
            {'label':'Disaster Rescue or Individual Tracking','value':'Disaster'},
            {'label':'Reset','value':'All'},
        ],
        value='All'

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         
#FIXED Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
        editable = True,
        row_selectable = "single", #user can select single row
        page_size=10,              #display 10 rows
        sort_action="native",      #allow for sorting
        filter_action="native",    #enable filter
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
    ])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback([Output('datatable-id','data'),
               Output('datatable-id','columns')],
              [Input('filter-type', 'value')]
             )
    
def update_dashboard(filter_type):
## FIXED Add code to filter interactive data table with MongoDB queries
    #'label':'Water Rescue','value':'Water'},
    #'label':'Mountain or Wilderness Rescue','value':'Mountain'},
    #'label':'Disaster Rescue or Individual Tracking','value':'Disaster'},
    #'label':'All','value':'All'},
#
#        
#       columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#      data=df.to_dict('records')
#       
#       
#      return (data,columns)

#to search for sequence of words (due to variations of mixed breeds), 
#adding regex (https://www.w3schools.com/python/python_regex.asp)
#https://docs.python.org/3/library/re.html

    #Water Rescue filter - includes: 
    #Breed: 'Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland'
    #Sex: 'Intact Female'
    #Training Age: '26 - 156 weeks'
    
    if filter_type == 'Water':
        
        lab = re.compile(".*lab.*",re.IGNORECASE)
        chesapeake = re.compile(".*chesa*.",re.IGNORECASE)
        newfound = re.compile(".*newfound*.",re.IGNORECASE)
   
        df = pd.DataFrame.from_records(db.read({
            '$or':[
                {"breed": {'$regex':lab}},
                {"breed": {'$regex':chesapeake}},
                {"breed": {'$regex':newfound}},
                #"breed": "Chesa Bay Retr", - this way doesn't work
            ],
            "sex_upon_outcome": "Intact Female",
            #https://www.w3schools.com/django/ref_lookups_gte.php for greater than/less than or equal to in weeks
            "age_upon_outcome_in_weeks": {"$gte":26.0, "$lte":156.0}
        
        }))
    #Mountain or Wilderness Rescue filter - includes:
    #Breed: "German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"
    #Sex: 'Intact Male'
    #Training Age: '26 - 156 weeks'
    elif filter_type == 'Mountain':
        
        german = re.compile(".*german.*",re.IGNORECASE)
        malamute = re.compile(".*malamute*.",re.IGNORECASE)
        sheepdog = re.compile(".*sheepdog*.",re.IGNORECASE)
        husky = re.compile(".*siberian*.",re.IGNORECASE)
        rott = re.compile(".*rott*.",re.IGNORECASE)
        
        df = pd.DataFrame.from_records(db.read({
            '$or':[
                {"breed": {'$regex':german}},
                {"breed": {'$regex':malamute}},
                {"breed": {'$regex':sheepdog}},
                {"breed": {'$regex':husky}},
                {"breed": {'$regex':rott}},                
            ],

            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte":26.0, "$lte":156.0}
        
        }))
    #Disaster or Individual Tracking filter - includes:
    #Breed: 'Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound','Rottweiler'
    #Sex: 'Intact Male'
    #Training Age: '20 - 300 weeks'
    elif filter_type == 'Disaster':
        
        german = re.compile(".*german.*",re.IGNORECASE)
        dober = re.compile(".*dober*.",re.IGNORECASE)
        gold = re.compile(".*gold*.",re.IGNORECASE)
        blood = re.compile(".*blood*.",re.IGNORECASE)
        rott = re.compile(".*rott*.",re.IGNORECASE)
        
        df = pd.DataFrame.from_records(db.read({
            '$or':[
                {"breed": {'$regex':dober}},
                {"breed": {'$regex':german}},
                {"breed": {'$regex':gold}},
                {"breed": {'$regex':blood}},
                {"breed": {'$regex':rott}},                
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte":20.0, "$lte":300.0}
        }))

    #Filter to return all animals
    elif filter_type == 'All':
        df=pd.DataFrame.from_records(db.read({}))
        
    #for errors
    else:
        raise Exception("Invalid filter")
    
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data=df.to_dict('records')
      
    return (data,columns)
    

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIXED ####
    # add code for chart of your choice (e.g. pie chart) #
    #https://www.geeksforgeeks.org/python/pie-plot-using-plotly-in-python/
    #https://plotly.com/python/pie-charts/
    
    #adding of pie chart
    
    df = pd.DataFrame.from_dict(viewData)
    
    return [
        dcc.Graph(            
            figure = px.pie(df, names='breed', title='Preferred Animals')
        )    
    ]
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return []

    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    #if no line is selected currently the map will not appear. Setting a default marker with tooltip and row to the first one
    if index is None or len(index) == 0:
        row = 0

    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13], dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run_server(debug=True)


Dash app running on http://127.0.0.1:29161/
